In [1]:
import sys

print(sys.executable)

c:\Users\Sushma2006\anaconda3\python.exe


In [2]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [3]:
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
# Load cleaned M5 dataset

df = pd.read_csv("../data/m5/m5_cleaned_CA1.csv")

df["date"] = pd.to_datetime(df["date"])

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (160360, 22)

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price']


C:\Users\Sushma2006\AppData\Local\Temp\ipykernel_13536\1768956134.py:3: DtypeWarning: Columns (16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/m5/m5_cleaned_CA1.csv")


In [5]:
# Convert date
df["date"] = pd.to_datetime(df["date"])

# Sort by product and date
df = df.sort_values(
    ["item_id", "store_id", "date"]
).reset_index(drop=True)

# Historical demand features
df["sales_lag_1"] = (
    df.groupby(["item_id", "store_id"])["sales"].shift(1)
)

df["sales_lag_7"] = (
    df.groupby(["item_id", "store_id"])["sales"].shift(7)
)

df["rolling_sales_7"] = (
    df.groupby(["item_id", "store_id"])["sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

# Historical price features
df["price_lag_1"] = (
    df.groupby(["item_id", "store_id"])["sell_price"].shift(1)
)

df["price_change"] = (
    df["sell_price"] - df["price_lag_1"]
)

df["price_change_pct"] = (
    df["price_change"] / df["price_lag_1"]
) * 100

# Calendar features
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Event features
df["has_event"] = (
    df["event_name_1"].notna() |
    df["event_name_2"].notna()
).astype(int)

df["event_name"] = (
    df["event_name_1"]
    .fillna(df["event_name_2"])
    .fillna("No Event")
)

df["event_type"] = (
    df["event_type_1"]
    .fillna(df["event_type_2"])
    .fillna("No Event")
)

# SNAP feature for California
df["snap"] = df["snap_CA"]

# Remove columns that are no longer needed
df = df.drop(
    columns=[
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2",
        "snap_CA",
        "snap_TX",
        "snap_WI"
    ]
)

print("Shape after feature engineering:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape after feature engineering: (160360, 29)

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'sell_price', 'sales_lag_1', 'sales_lag_7', 'rolling_sales_7', 'price_lag_1', 'price_change', 'price_change_pct', 'day_of_week', 'day_of_month', 'week_of_year', 'is_weekend', 'has_event', 'event_name', 'event_type', 'snap']


In [6]:
df = df.dropna(
    subset=[
        "sales_lag_1",
        "sales_lag_7",
        "rolling_sales_7",
        "price_lag_1"
    ]
).copy()

print("Final training dataset shape:", df.shape)

print("\nMissing values:")
print(
    df[
        [
            "sales_lag_1",
            "sales_lag_7",
            "rolling_sales_7",
            "price_lag_1",
            "price_change_pct"
        ]
    ].isna().sum()
)

Final training dataset shape: (159660, 29)

Missing values:
sales_lag_1         0
sales_lag_7         0
rolling_sales_7     0
price_lag_1         0
price_change_pct    0
dtype: int64


In [7]:
# Select features for the demand forecasting model

features = [
    "item_id",
    "dept_id",
    "cat_id",
    "sell_price",
    "sales_lag_1",
    "sales_lag_7",
    "rolling_sales_7",
    "price_lag_1",
    "price_change",
    "price_change_pct",
    "day_of_week",
    "day_of_month",
    "month",
    "year",
    "week_of_year",
    "is_weekend",
    "has_event",
    "event_name",
    "event_type",
    "snap"
]

target = "sales"

X = df[features].copy()
y = df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (159660, 20)
y shape: (159660,)


In [8]:
categorical_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "event_name",
    "event_type"
]

X_encoded = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True
)

print("Encoded X shape:", X_encoded.shape)

Encoded X shape: (159660, 148)


In [9]:
# Find the date at the 80% point

split_date = df["date"].quantile(0.80)

print("Split date:", split_date)

Split date: 2015-06-10 00:00:00


In [10]:
# Create time-based train/test masks

train_mask = df["date"] < split_date
test_mask = df["date"] >= split_date

X_train = X_encoded.loc[train_mask].copy()
X_test = X_encoded.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining date range:")
print(df.loc[train_mask, "date"].min())
print("to")
print(df.loc[train_mask, "date"].max())

print("\nTesting date range:")
print(df.loc[test_mask, "date"].min())
print("to")
print(df.loc[test_mask, "date"].max())

Training samples: 127660
Testing samples: 32000

Training date range:
2011-02-05 00:00:00
to
2015-06-09 00:00:00

Testing date range:
2015-06-10 00:00:00
to
2016-04-24 00:00:00


In [11]:
# Train baseline XGBoost demand forecasting model

baseline_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(X_train, y_train)

print("Baseline XGBoost model training completed.")

Baseline XGBoost model training completed.


In [12]:
# Make predictions

y_pred = baseline_model.predict(X_test)

# Calculate evaluation metrics

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Baseline XGBoost Results")
print("------------------------")
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

Baseline XGBoost Results
------------------------
MAE : 1.1376316547393799
MSE : 7.161227703094482
RMSE: 2.676047029312916
R²  : 0.35198521614074707


In [13]:
print("Demand statistics:")
print(y.describe())

print("\nActual vs predicted:")
print(
    pd.DataFrame({
        "actual": y_test.values[:20],
        "predicted": y_pred[:20]
    })
)

Demand statistics:
count    159660.000000
mean          1.260967
std           3.335737
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          91.000000
Name: sales, dtype: float64

Actual vs predicted:
    actual  predicted
0        0   0.304162
1        0   0.304162
2        0   0.369325
3        0   0.445324
4        0   0.465974
5        0   0.260242
6        0   0.099017
7        0   0.101802
8        1   0.099017
9        3   0.364854
10       1   0.715787
11       1   0.831055
12       0   0.666165
13       1   0.618954
14       1   0.792733
15       0   0.882649
16       0   0.922361
17       0   0.676339
18       1   0.549503
19       1   0.412941


In [14]:
importance = pd.Series(
    baseline_model.feature_importances_,
    index=X_encoded.columns
).sort_values(ascending=False)

print("Top 20 important features:")
print(importance.head(20))

Top 20 important features:
rolling_sales_7                   0.124484
price_lag_1                       0.094296
sell_price                        0.034185
event_name_Mother's day           0.030263
event_name_Cinco De Mayo          0.029861
item_id_HOBBIES_1_015             0.028287
item_id_HOBBIES_1_048             0.028267
item_id_HOBBIES_1_080             0.022452
event_name_PresidentsDay          0.021081
event_name_Eid al-Fitr            0.021051
item_id_HOBBIES_1_032             0.020617
item_id_HOBBIES_1_043             0.019850
event_type_National               0.018160
event_name_Ramadan starts         0.018024
event_name_MartinLutherKingDay    0.017984
year                              0.016580
event_name_Halloween              0.016404
item_id_HOBBIES_1_016             0.015966
event_name_LentWeek2              0.015422
item_id_HOBBIES_1_067             0.015320
dtype: float32


In [15]:
# Log-transform the demand target

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print("Original demand:")
print(y_train.describe())

print("\nLog-transformed demand:")
print(y_train_log.describe())

Original demand:
count    127660.000000
mean          1.266223
std           3.338575
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          91.000000
Name: sales, dtype: float64

Log-transformed demand:
count    127660.000000
mean          0.475175
std           0.683055
min           0.000000
25%           0.000000
50%           0.000000
75%           0.693147
max           4.521789
Name: sales, dtype: float64


In [16]:
# XGBoost model using log-transformed demand

log_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

log_model.fit(X_train, y_train_log)

print("Log-transformed XGBoost training completed.")

Log-transformed XGBoost training completed.


In [17]:
# Predict and convert back to original sales scale

y_pred_log = log_model.predict(X_test)

y_pred_original = np.expm1(y_pred_log)

# Demand cannot be negative
y_pred_original = np.maximum(y_pred_original, 0)

mae_log = mean_absolute_error(y_test, y_pred_original)
mse_log = mean_squared_error(y_test, y_pred_original)
rmse_log = np.sqrt(mse_log)
r2_log = r2_score(y_test, y_pred_original)

print("Log-XGBoost Results")
print("-------------------")
print("MAE :", mae_log)
print("MSE :", mse_log)
print("RMSE:", rmse_log)
print("R²  :", r2_log)

Log-XGBoost Results
-------------------
MAE : 1.0791805982589722
MSE : 8.183731079101562
RMSE: 2.8607221254609057
R²  : 0.25945955514907837


In [18]:
product_demand = (
    df.groupby("item_id")["sales"]
    .agg(["count", "sum", "mean", "max"])
    .sort_values("mean", ascending=False)
)

print(product_demand.head(20))

               count    sum      mean  max
item_id                                   
HOBBIES_1_048   1787  13866  7.759373   67
HOBBIES_1_008   1906  13793  7.236621   91
HOBBIES_1_019   1381   9025  6.535119   64
HOBBIES_1_015   1906  11593  6.082371   83
HOBBIES_1_032   1906  10569  5.545121   72
HOBBIES_1_016   1906  10518  5.518363   90
HOBBIES_1_067   1906   9892  5.189927   81
HOBBIES_1_103   1906   7858  4.122770   43
HOBBIES_1_030   1367   5592  4.090710   52
HOBBIES_1_080   1906   6566  3.444911   66
HOBBIES_1_043   1892   6263  3.310254   58
HOBBIES_1_090   1031   3176  3.080504   39
HOBBIES_1_078   1906   3723  1.953305   11
HOBBIES_1_074   1605   3060  1.906542   13
HOBBIES_1_004   1871   3284  1.755211   15
HOBBIES_1_055   1906   3201  1.679433   11
HOBBIES_1_089   1906   3051  1.600735   11
HOBBIES_1_064   1899   2913  1.533965    8
HOBBIES_1_029   1906   2804  1.471144   11
HOBBIES_1_014   1647   2397  1.455373   15


In [19]:
print("Number of products:", df["item_id"].nunique())

print("\nAverage demand by product:")
print(product_demand["mean"].describe())

Number of products: 100

Average demand by product:
count    100.000000
mean       1.179790
std        1.627655
min        0.021924
25%        0.320402
50%        0.589718
75%        1.133395
max        7.759373
Name: mean, dtype: float64


In [20]:
# Create week identifier

df["week_start"] = (
    df["date"]
    - pd.to_timedelta(df["date"].dt.dayofweek, unit="D")
)

print(df[["date", "week_start", "sales"]].head(10))

         date week_start  sales
7  2013-07-20 2013-07-15      0
8  2013-07-21 2013-07-15      0
9  2013-07-22 2013-07-22      0
10 2013-07-23 2013-07-22      0
11 2013-07-24 2013-07-22      0
12 2013-07-25 2013-07-22      0
13 2013-07-26 2013-07-22      0
14 2013-07-27 2013-07-22      0
15 2013-07-28 2013-07-22      0
16 2013-07-29 2013-07-29      0


In [21]:
# Aggregate daily demand into weekly demand

weekly_df = (
    df.groupby(
        [
            "item_id",
            "dept_id",
            "cat_id",
            "week_start"
        ],
        as_index=False
    )
    .agg(
        weekly_sales=("sales", "sum"),
        avg_price=("sell_price", "mean"),
        min_price=("sell_price", "min"),
        max_price=("sell_price", "max")
    )
)

print("Weekly dataset shape:", weekly_df.shape)

print("\nWeekly demand statistics:")
print(weekly_df["weekly_sales"].describe())

Weekly dataset shape: (22880, 8)

Weekly demand statistics:
count    22880.000000
mean         8.799213
std         15.241299
min          0.000000
25%          1.000000
50%          4.000000
75%          9.000000
max        172.000000
Name: weekly_sales, dtype: float64


In [22]:
weekly_df["price_vs_avg"] = (
    weekly_df["avg_price"] /
    weekly_df.groupby("item_id")["avg_price"].transform("mean")
)

weekly_df["price_change_abs"] = (
    weekly_df["avg_price"] -
    weekly_df.groupby("item_id")["avg_price"].transform("mean")
)

In [23]:
# Sort by product and week

weekly_df = weekly_df.sort_values(
    ["item_id", "week_start"]
).reset_index(drop=True)

# Previous week's demand
weekly_df["sales_lag_1"] = (
    weekly_df.groupby("item_id")["weekly_sales"].shift(1)
)

# Demand from 2 weeks ago
weekly_df["sales_lag_2"] = (
    weekly_df.groupby("item_id")["weekly_sales"].shift(2)
)

# Demand from 4 weeks ago
weekly_df["sales_lag_4"] = (
    weekly_df.groupby("item_id")["weekly_sales"].shift(4)
)

# 4-week rolling average
weekly_df["rolling_sales_4"] = (
    weekly_df.groupby("item_id")["weekly_sales"]
    .transform(
        lambda x: x.shift(1).rolling(4).mean()
    )
)

print(weekly_df.head(15))

          item_id    dept_id   cat_id week_start  weekly_sales  avg_price  \
0   HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-07-15             0   9.580000   
1   HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-07-22             0   9.202857   
2   HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-07-29             3   8.260000   
3   HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-08-05             2   8.260000   
4   HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-08-12             5   8.260000   
5   HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-08-19             1   8.260000   
6   HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-08-26             2   8.260000   
7   HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-09-02             5   8.260000   
8   HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-09-09             1   8.260000   
9   HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-09-16             6   8.260000   
10  HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-09-23             1   8.260000   
11  HOBBIES_1_001  HOBBIES_1  HOBBIES 2013-09-30             5   8.260000   

In [24]:
weekly_df = weekly_df.dropna(
    subset=[
        "sales_lag_1",
        "sales_lag_2",
        "sales_lag_4",
        "rolling_sales_4"
    ]
).copy()

print("Weekly dataset after lag features:", weekly_df.shape)

print("\nMissing values:")
print(
    weekly_df[
        [
            "sales_lag_1",
            "sales_lag_2",
            "sales_lag_4",
            "rolling_sales_4"
        ]
    ].isna().sum()
)

Weekly dataset after lag features: (22480, 14)

Missing values:
sales_lag_1        0
sales_lag_2        0
sales_lag_4        0
rolling_sales_4    0
dtype: int64


In [25]:
# Add calendar features

weekly_df["month"] = weekly_df["week_start"].dt.month
weekly_df["year"] = weekly_df["week_start"].dt.year
weekly_df["week_of_year"] = (
    weekly_df["week_start"]
    .dt.isocalendar()
    .week
    .astype(int)
)

weekly_df["quarter"] = (
    weekly_df["week_start"].dt.quarter
)

print(
    weekly_df[
        [
            "week_start",
            "month",
            "year",
            "week_of_year",
            "quarter"
        ]
    ].head(10)
)

   week_start  month  year  week_of_year  quarter
4  2013-08-12      8  2013            33        3
5  2013-08-19      8  2013            34        3
6  2013-08-26      8  2013            35        3
7  2013-09-02      9  2013            36        3
8  2013-09-09      9  2013            37        3
9  2013-09-16      9  2013            38        3
10 2013-09-23      9  2013            39        3
11 2013-09-30      9  2013            40        3
12 2013-10-07     10  2013            41        4
13 2013-10-14     10  2013            42        4


In [26]:
# Weekly price change

weekly_df["price_change"] = (
    weekly_df["avg_price"]
    - weekly_df.groupby("item_id")["avg_price"].shift(1)
)

weekly_df["price_change_pct"] = (
    weekly_df["price_change"]
    / weekly_df.groupby("item_id")["avg_price"].shift(1)
) * 100

print(
    weekly_df[
        [
            "week_start",
            "item_id",
            "avg_price",
            "price_change",
            "price_change_pct"
        ]
    ].head(15)
)

   week_start        item_id  avg_price  price_change  price_change_pct
4  2013-08-12  HOBBIES_1_001       8.26           NaN               NaN
5  2013-08-19  HOBBIES_1_001       8.26           0.0               0.0
6  2013-08-26  HOBBIES_1_001       8.26           0.0               0.0
7  2013-09-02  HOBBIES_1_001       8.26           0.0               0.0
8  2013-09-09  HOBBIES_1_001       8.26           0.0               0.0
9  2013-09-16  HOBBIES_1_001       8.26           0.0               0.0
10 2013-09-23  HOBBIES_1_001       8.26           0.0               0.0
11 2013-09-30  HOBBIES_1_001       8.26           0.0               0.0
12 2013-10-07  HOBBIES_1_001       8.26           0.0               0.0
13 2013-10-14  HOBBIES_1_001       8.26           0.0               0.0
14 2013-10-21  HOBBIES_1_001       8.26           0.0               0.0
15 2013-10-28  HOBBIES_1_001       8.26           0.0               0.0
16 2013-11-04  HOBBIES_1_001       8.26           0.0           

In [27]:
print("Missing price-change values:")
print(weekly_df["price_change_pct"].isna().sum())

Missing price-change values:
100


In [28]:
# Fill missing price-change values for the first week of each product

weekly_df["price_change"] = weekly_df["price_change"].fillna(0)
weekly_df["price_change_pct"] = (
    weekly_df["price_change_pct"].fillna(0)
)

print("Missing price-change values:")
print(weekly_df["price_change"].isna().sum())
print(weekly_df["price_change_pct"].isna().sum())

Missing price-change values:
0
0


In [29]:
print("Final weekly dataset shape:", weekly_df.shape)

print("\nColumns:")
print(weekly_df.columns.tolist())

Final weekly dataset shape: (22480, 20)

Columns:
['item_id', 'dept_id', 'cat_id', 'week_start', 'weekly_sales', 'avg_price', 'min_price', 'max_price', 'price_vs_avg', 'price_change_abs', 'sales_lag_1', 'sales_lag_2', 'sales_lag_4', 'rolling_sales_4', 'month', 'year', 'week_of_year', 'quarter', 'price_change', 'price_change_pct']


In [30]:
# Prepare weekly event and SNAP information

weekly_extra = (
    df.groupby(
        ["item_id", "week_start"],
        as_index=False
    )
    .agg(
        has_event=("has_event", "max"),
        snap=("snap", "max")
    )
)

print(weekly_extra.head(10))

         item_id week_start  has_event  snap
0  HOBBIES_1_001 2013-07-15          0     0
1  HOBBIES_1_001 2013-07-22          0     0
2  HOBBIES_1_001 2013-07-29          0     1
3  HOBBIES_1_001 2013-08-05          1     1
4  HOBBIES_1_001 2013-08-12          0     0
5  HOBBIES_1_001 2013-08-19          0     0
6  HOBBIES_1_001 2013-08-26          0     1
7  HOBBIES_1_001 2013-09-02          1     1
8  HOBBIES_1_001 2013-09-09          0     1
9  HOBBIES_1_001 2013-09-16          0     0


In [31]:
# Merge event and SNAP features

weekly_df = weekly_df.merge(
    weekly_extra,
    on=["item_id", "week_start"],
    how="left"
)

print("Weekly dataset shape:", weekly_df.shape)

print("\nMissing values:")
print(
    weekly_df[
        ["has_event", "snap"]
    ].isna().sum()
)

Weekly dataset shape: (22480, 22)

Missing values:
has_event    0
snap         0
dtype: int64


In [32]:
print("\nEvent distribution:")
print(weekly_df["has_event"].value_counts())

print("\nSNAP distribution:")
print(weekly_df["snap"].value_counts())


Event distribution:
has_event
0    11851
1    10629
Name: count, dtype: int64

SNAP distribution:
snap
1    11847
0    10633
Name: count, dtype: int64


In [33]:
# Select final features for weekly demand forecasting

weekly_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "avg_price",
    "min_price",
    "max_price",
    "sales_lag_1",
    "sales_lag_2",
    "sales_lag_4",
    "rolling_sales_4",
    "price_change",
    "price_change_pct",
    "month",
    "year",
    "week_of_year",
    "quarter",
    "has_event",
    "snap"
]

weekly_target = "weekly_sales"

X_weekly = weekly_df[weekly_features].copy()
y_weekly = weekly_df[weekly_target].copy()

print("X weekly shape:", X_weekly.shape)
print("y weekly shape:", y_weekly.shape)

X weekly shape: (22480, 18)
y weekly shape: (22480,)


In [34]:
categorical_features = [
    "item_id",
    "dept_id",
    "cat_id"
]

X_weekly_encoded = pd.get_dummies(
    X_weekly,
    columns=categorical_features,
    drop_first=True
)

print("Encoded weekly X shape:", X_weekly_encoded.shape)

Encoded weekly X shape: (22480, 114)


In [35]:
# Find the 80% time split

weekly_split_date = weekly_df["week_start"].quantile(0.80)

print("Split date:", weekly_split_date)

Split date: 2015-06-15 00:00:00


In [36]:
# Create time-based train/test masks

train_mask_weekly = (
    weekly_df["week_start"] < weekly_split_date
)

test_mask_weekly = (
    weekly_df["week_start"] >= weekly_split_date
)

X_train_weekly = X_weekly_encoded.loc[
    train_mask_weekly
].copy()

X_test_weekly = X_weekly_encoded.loc[
    test_mask_weekly
].copy()

y_train_weekly = y_weekly.loc[
    train_mask_weekly
].copy()

y_test_weekly = y_weekly.loc[
    test_mask_weekly
].copy()

print("Training samples:", len(X_train_weekly))
print("Testing samples:", len(X_test_weekly))

print("\nTraining date range:")
print(weekly_df.loc[train_mask_weekly, "week_start"].min())
print("to")
print(weekly_df.loc[train_mask_weekly, "week_start"].max())

print("\nTesting date range:")
print(weekly_df.loc[test_mask_weekly, "week_start"].min())
print("to")
print(weekly_df.loc[test_mask_weekly, "week_start"].max())

Training samples: 17980
Testing samples: 4500

Training date range:


2011-02-28 00:00:00
to
2015-06-08 00:00:00

Testing date range:
2015-06-15 00:00:00
to
2016-04-18 00:00:00


In [37]:
# Train baseline XGBoost model for weekly demand

weekly_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

weekly_model.fit(
    X_train_weekly,
    y_train_weekly
)

print("Weekly baseline model training completed.")

Weekly baseline model training completed.


In [38]:
# Predict weekly demand

y_pred_weekly = weekly_model.predict(
    X_test_weekly
)

# Calculate metrics

mae_weekly = mean_absolute_error(
    y_test_weekly,
    y_pred_weekly
)

mse_weekly = mean_squared_error(
    y_test_weekly,
    y_pred_weekly
)

rmse_weekly = np.sqrt(mse_weekly)

r2_weekly = r2_score(
    y_test_weekly,
    y_pred_weekly
)

print("Weekly XGBoost Results")
print("----------------------")
print("MAE :", mae_weekly)
print("MSE :", mse_weekly)
print("RMSE:", rmse_weekly)
print("R²  :", r2_weekly)

Weekly XGBoost Results
----------------------
MAE : 3.5271177291870117
MSE : 57.235389709472656
RMSE: 7.565407438431367
R²  : 0.7755106687545776


In [39]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=xgb.XGBRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(
    X_train_weekly,
    y_train_weekly
)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest CV MSE:")
print(-grid_search.best_score_)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best parameters:
{'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.8}

Best CV MSE:
70.84306335449219


In [40]:
# Get the best tuned model

best_weekly_model = grid_search.best_estimator_

# Predict on the unseen test period

y_pred_tuned = best_weekly_model.predict(
    X_test_weekly
)

# Calculate evaluation metrics

mae_tuned = mean_absolute_error(
    y_test_weekly,
    y_pred_tuned
)

mse_tuned = mean_squared_error(
    y_test_weekly,
    y_pred_tuned
)

rmse_tuned = np.sqrt(mse_tuned)

r2_tuned = r2_score(
    y_test_weekly,
    y_pred_tuned
)

print("Tuned Weekly XGBoost Results")
print("----------------------------")
print("MAE :", mae_tuned)
print("MSE :", mse_tuned)
print("RMSE:", rmse_tuned)
print("R²  :", r2_tuned)

Tuned Weekly XGBoost Results
----------------------------
MAE : 3.5378923416137695
MSE : 55.28144836425781
RMSE: 7.435149518621519
R²  : 0.783174455165863


In [41]:
print("\nModel Comparison")
print("----------------")
print("Baseline R²:", r2_weekly)
print("Tuned R²   :", r2_tuned)

print("\nBaseline RMSE:", rmse_weekly)
print("Tuned RMSE   :", rmse_tuned)



Model Comparison
----------------
Baseline R²: 0.7755106687545776
Tuned R²   : 0.783174455165863

Baseline RMSE: 7.565407438431367
Tuned RMSE   : 7.435149518621519


In [42]:
# Save the final tuned model

MODEL_PATH = "../models/m5_weekly_demand_model.json"
FEATURES_PATH = "../models/m5_weekly_model_features.pkl"

best_weekly_model.save_model(MODEL_PATH)

joblib.dump(
    X_weekly_encoded.columns.tolist(),
    FEATURES_PATH
)

print("Final model saved successfully.")
print("Model:", MODEL_PATH)
print("Features:", FEATURES_PATH)

Final model saved successfully.
Model: ../models/m5_weekly_demand_model.json
Features: ../models/m5_weekly_model_features.pkl


In [43]:
import os

print("Model exists:", os.path.exists(MODEL_PATH))
print("Features exist:", os.path.exists(FEATURES_PATH))

Model exists: True
Features exist: True


In [44]:
# Load the saved model again

loaded_m5_model = xgb.XGBRegressor()

loaded_m5_model.load_model(
    "../models/m5_weekly_demand_model.json"
)

loaded_features = joblib.load(
    "../models/m5_weekly_model_features.pkl"
)

print("Saved model loaded successfully.")
print("Number of features:", len(loaded_features))

Saved model loaded successfully.
Number of features: 114


In [45]:
# Test the reloaded model on the test dataset

reloaded_predictions = loaded_m5_model.predict(
    X_test_weekly
)

reloaded_mae = mean_absolute_error(
    y_test_weekly,
    reloaded_predictions
)

reloaded_rmse = np.sqrt(
    mean_squared_error(
        y_test_weekly,
        reloaded_predictions
    )
)

reloaded_r2 = r2_score(
    y_test_weekly,
    reloaded_predictions
)

print("Reloaded Model Results")
print("----------------------")
print("MAE :", reloaded_mae)
print("RMSE:", reloaded_rmse)
print("R²  :", reloaded_r2)

Reloaded Model Results
----------------------
MAE : 3.5378923416137695
RMSE: 7.435149518621519
R²  : 0.783174455165863
